# Lab: Jobs & Workflows

**Course 1, Week 3: Delta Lake & Workflows**

## Objectives
- Build a parameterized ETL notebook
- Use widgets for runtime parameters
- Create dashboard-ready queries
- Understand job scheduling concepts


## Part 1: Parameterized Notebook

EXERCISE: Create widgets and use them in your pipeline.

In [0]:
# EXERCISE: Create two widgets:
# 1. "start_date" (text widget, default "2024-01-01")
# 2. "category_filter" (dropdown widget, options: "All", "Electronics", "Books", "Clothing")
# YOUR CODE HERE

# Hint:
# dbutils.widgets.text("start_date", "2024-01-01", "Start Date")
# dbutils.widgets.dropdown("category_filter", "All", ["All", "Electronics", "Books", "Clothing"], "Category")

# Create two widgets: text and dropdown
dbutils.widgets.text("start_date", "2024-01-01", "Start Date")
dbutils.widgets.dropdown("category_filter", "All", ["All", "Electronics", "Books", "Clothing"], "Category")


In [0]:

# EXERCISE: Read the widget values into variables
# YOUR CODE HERE
# start_date = dbutils.widgets.get("start_date")
# category_filter = dbutils.widgets.get("category_filter")

# Fallback for non-Databricks environments
try:
    start_date = dbutils.widgets.get("start_date")
    category_filter = dbutils.widgets.get("category_filter")
except NameError:
    start_date = "2024-01-01"
    category_filter = "All"

print(f"Parameters: start_date={start_date}, category_filter={category_filter}")

Parameters: start_date=2024-01-01, category_filter=All


## Part 2: Build an ETL Pipeline

Implement Extract, Transform, Load steps.

In [0]:
from pyspark.sql import functions as F

# EXTRACT: Raw order data
raw_orders = spark.createDataFrame(
    [
        ("2024-01-01", "O001", "Electronics", "Laptop", 999.99, 1, "completed"),
        ("2024-01-01", "O002", "Books", "Python Guide", 49.99, 2, "completed"),
        ("2024-01-02", "O003", "Electronics", "Phone", 699.99, 1, "cancelled"),
        ("2024-01-02", "O004", "Clothing", "Jacket", 129.99, 3, "completed"),
        ("2024-01-03", "O005", "Books", "ML Handbook", 69.99, 1, "completed"),
        ("2024-01-03", "O006", "Electronics", "Tablet", 449.99, 2, "completed"),
        ("2024-01-04", "O007", "Clothing", "Shoes", 89.99, 2, "completed"),
        ("2024-01-04", "O008", "Electronics", "Earbuds", 79.99, 5, "completed"),
        ("2024-01-05", "O009", "Books", "Data Science", 59.99, 3, "pending"),
        ("2024-01-05", "O010", "Clothing", "Hat", 29.99, 4, "completed"),
    ],
    ["order_date", "order_id", "category", "product", "price", "quantity", "status"],
)

print(f"Extracted {raw_orders.count()} raw orders")


Extracted 10 raw orders


In [0]:
# EXERCISE: TRANSFORM the data
# 1. Filter to only "completed" orders
# 2. Filter by start_date (order_date >= start_date)
# 3. Filter by category_filter (if not "All")
# 4. Add a "revenue" column (price * quantity)
# 5. Add a "processed_at" timestamp column
# YOUR CODE HERE


# 1. Filter to only "completed" orders and by start_date
transformed_orders = raw_orders.filter(
    (F.col("status") == "completed") & 
    (F.col("order_date") >= start_date)
)

# 3. Filter by category_filter (if not "All")
if category_filter != "All":
    transformed_orders = transformed_orders.filter(F.col("category") == category_filter)

# 4 & 5. Add revenue and processed_at timestamp columns
transformed_orders = transformed_orders \
    .withColumn("revenue", F.col("price") * F.col("quantity")) \
    .withColumn("processed_at", F.current_timestamp())

display(transformed_orders)

order_date,order_id,category,product,price,quantity,status,revenue,processed_at
2024-01-01,O001,Electronics,Laptop,999.99,1,completed,999.99,2026-09-02T04:41:16.703Z
2024-01-01,O002,Books,Python Guide,49.99,2,completed,99.98,2026-09-02T04:41:16.703Z
2024-01-02,O004,Clothing,Jacket,129.99,3,completed,389.97,2026-09-02T04:41:16.703Z
2024-01-03,O005,Books,ML Handbook,69.99,1,completed,69.99,2026-09-02T04:41:16.703Z
2024-01-03,O006,Electronics,Tablet,449.99,2,completed,899.98,2026-09-02T04:41:16.703Z
2024-01-04,O007,Clothing,Shoes,89.99,2,completed,179.98,2026-09-02T04:41:16.703Z
2024-01-04,O008,Electronics,Earbuds,79.99,5,completed,399.95,2026-09-02T04:41:16.703Z
2024-01-05,O010,Clothing,Hat,29.99,4,completed,119.96,2026-09-02T04:41:16.703Z


In [0]:
# EXERCISE: LOAD the transformed data into a Delta table "lab_orders_gold"
# Use overwrite mode
# YOUR CODE HERE


transformed_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lab_orders_gold")

print(f"Successfully loaded {transformed_orders.count()} records into lab_orders_gold")


Successfully loaded 8 records into lab_orders_gold


## Part 3: Dashboard Queries

EXERCISE: Write queries that could power a dashboard.

In [0]:
%sql
-- EXERCISE: Revenue by category (for a bar chart)
-- Columns: category, total_revenue, order_count
-- YOUR CODE HERE

SELECT 
    category, 
    SUM(revenue) AS total_revenue, 
    COUNT(order_id) AS order_count
FROM lab_orders_gold
GROUP BY category;

category,total_revenue,order_count
Electronics,2299.92,3
Books,169.97,2
Clothing,689.9100000000001,3


In [0]:
%sql
-- EXERCISE: Daily revenue trend (for a line chart)
-- Columns: order_date, daily_revenue, cumulative_revenue
-- YOUR CODE HERE (Hint: Use SUM() OVER(ORDER BY order_date) for cumulative)

SELECT 
    order_date, 
    SUM(revenue) AS daily_revenue,
    SUM(SUM(revenue)) OVER (ORDER BY order_date) AS cumulative_revenue
FROM lab_orders_gold
GROUP BY order_date
ORDER BY order_date;

order_date,daily_revenue,cumulative_revenue
2024-01-01,1099.97,1099.97
2024-01-02,389.97,1489.94
2024-01-03,969.97,2459.91
2024-01-04,579.93,3039.8399999999997
2024-01-05,119.96,3159.7999999999997


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- EXERCISE: Top products by revenue (for a table/leaderboard)
-- Columns: product, category, total_revenue, total_quantity
-- Order by total_revenue DESC, limit 5
-- YOUR CODE HERE

SELECT 
    product, 
    category, 
    SUM(revenue) AS total_revenue, 
    SUM(quantity) AS total_quantity
FROM lab_orders_gold
GROUP BY product, category
ORDER BY total_revenue DESC
LIMIT 5;

product,category,total_revenue,total_quantity
Laptop,Electronics,999.99,1
Tablet,Electronics,899.98,2
Earbuds,Electronics,399.95,5
Jacket,Clothing,389.97,3
Shoes,Clothing,179.98,2


## Part 4: Job Configuration (Conceptual)

EXERCISE: Answer these questions about scheduling this notebook as a job.

**Q1:** What cluster type should you use for a scheduled daily job?

**Your answer:** Job Cluster. Job clusters spin up specifically for the run and terminate automatically, making them significantly more cost-effective than keeping an All-Purpose interactive cluster running.

**Q2:** If this job fails, what retry configuration would you set?

**Your answer:** Automatic retries, such as setting 2 to 3 retries with a short interval or backoff. This handles transient issues like temporary network glitches or brief source database downtime without requiring manual intervention.

**Q3:** How would you pass the start_date parameter when running as a job?

**Your answer:** Task parameters or job workflow parameters. You configure key-value pairs in the Databricks Workflows task settings, which are then read inside the notebook using widgets like dbutils.widgets.get("start_date").

## Validation

In [0]:
def validate_lab():
    """Validate lab completion."""
    checks = []

    # Check 1: Parameters set
    checks.append(("Parameters configured", start_date is not None and category_filter is not None))

    # Check 2: Gold table exists
    try:
        df = spark.sql("SELECT * FROM lab_orders_gold")
        checks.append(("Gold table created", df.count() > 0))
    except Exception:
        checks.append(("Gold table created", False))
        df = None

    # Check 3: Revenue column exists
    if df:
        checks.append(("Revenue column", "revenue" in df.columns))

    # Check 4: Only completed orders
    if df:
        statuses = [row.status for row in df.select("status").distinct().collect()]
        checks.append(("Filtered to completed", statuses == ["completed"]))

    print("Lab Validation Results:")
    print("-" * 40)
    all_passed = True
    for name, passed in checks:
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {name}")
        if not passed:
            all_passed = False

    if all_passed:
        print("\nAll checks passed! Lab complete.")
    else:
        print("\nSome checks failed. Review your code above.")

validate_lab()

Lab Validation Results:
----------------------------------------
  [PASS] Parameters configured
  [PASS] Gold table created
  [PASS] Revenue column
  [PASS] Filtered to completed

All checks passed! Lab complete.


In [0]:
# Clean up
try:
    spark.sql("DROP TABLE IF EXISTS lab_orders_gold")
    dbutils.widgets.removeAll()
except Exception:
    pass


#### Databricks Jobs

A **Job** is a scheduled or triggered execution of a notebook, script, or JAR:

| Feature | Description |
|---------|-------------|
| **Schedule** | Cron-based or event-triggered |
| **Cluster** | Uses job clusters (auto-created, auto-terminated) |
| **Retries** | Configurable retry on failure |
| **Alerts** | Email/webhook notifications |
| **Parameters** | Pass runtime parameters to notebooks |

 Jobs are created via the **Workflows** UI or the **Jobs API**.


#### Multi-Step Workflows

Databricks Workflows orchestrate multiple tasks:

```
Workflow: Daily Sales Pipeline
├── Task 1: Extract (notebook: extract_raw_data)
├── Task 2: Transform (notebook: clean_and_enrich)
│   └── depends_on: Task 1
├── Task 3: Load (notebook: write_to_delta)
│   └── depends_on: Task 2
└── Task 4: Quality Check (notebook: validate_output)
    └── depends_on: Task 3
```

**Workflow features:**
- **DAG execution:** Tasks run in dependency order
- **Conditional logic:** Run tasks based on previous results
- **Error handling:** Retry policies per task
- **Cluster reuse:** Share clusters across tasks


#### Monitoring and Alerts

| Feature | Purpose | Configuration |
|---------|---------|---------------|
| **Run History** | View past job executions | Workflows UI |
| **Email Alerts** | Notify on success/failure | Job settings |
| **Webhook** | Trigger external systems | Job settings |
| **Ganglia Metrics** | Cluster performance | Compute UI |
| **Query History** | SQL execution tracking | SQL Warehouse UI |


#### Creating a Job (UI Steps)

To create a job in Databricks:

1. Navigate to **Workflows** in the left sidebar
2. Click **Create Job**
3. Name your job (e.g., "Daily Sales ETL")
4. Add a task:
- Select task type: **Notebook**
- Choose this notebook
- Select cluster: **Job Cluster** (recommended) or existing cluster
5. Set schedule: e.g., daily at 6:00 AM UTC
6. Configure alerts: email on failure
7. Click **Create**

#### Summary

| Component | Purpose | Key Feature |
|-----------|---------|-------------|
| **Jobs** | Scheduled notebook execution | Cron schedules, retries |
| **Dashboards** | SQL-based visualizations | Connected to SQL Warehouse |
| **Workflows** | Multi-task orchestration | DAG dependencies |
| **Widgets** | Parameterized notebooks | Reusable pipelines |